# 00. Подготовка корпуса LatinISE

    **Входные данные:** готовые лемматизированные тексты в `data/new_lemmatized_texts/` и таблица `data/metadata.csv`.

    Мы считаем, что тексты уже исправлены. Ноутбук **не меняет** их, не проводит лемматизацию и не проверяет отдельных слов.
    Его задача — распределить произведения по периодам и показать размер корпуса.

In [1]:
from pathlib import Path
import sys

# При необходимости укажите каталог на Google Drive:
PROJECT_ROOT = None  # например: '/content/drive/MyDrive/latin-embeddings-models'

candidates = [Path(PROJECT_ROOT)] if PROJECT_ROOT else [Path.cwd(), Path.cwd().parent, Path.cwd() / 'latin-embeddings-models']
root = next((p.resolve() for p in candidates if (p / 'src/core.py').exists()), None)
if root is None:
    raise FileNotFoundError('Укажите PROJECT_ROOT — папку репозитория.')
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print('Проект:', root)

Проект: C:\Users\Admin\Documents\GitHub\latinise-humanities


## 1. Состав двух хронологических срезов

    Произведения с датой окончания **до 180 г.** относятся к раннему периоду, произведения с датой окончания **180–605 гг.** — к позднему. Произведения с пометкой `data_issue` исключаются. При наличии датировки, пересекающей границу, действует то же правило по году окончания.

In [2]:
from src import core
from IPython.display import display
df = core.metadata()
display(df.groupby('slice').agg(произведений=('file', 'size')).reset_index())

,slice,произведений
0,post180,391
1,pre180,387


## 2. Подсчёт характеристик корпуса

    Проверяем только наличие необходимых файлов. Для каждого периода считаем произведения, строки с леммами, словоупотребления и размер словаря. При чтении токены приводятся к нижнему регистру, `v` заменяется на `u`, краевая пунктуация удаляется — как и в обучении. **Исходные тексты не переписываются.**

In [3]:
summary = core.prepare()
display(summary)

,period,texts,sentences,tokens,lemmas,min_count_5,min_count_50
0,pre180,387,194612,3569242,61409,18806,5548
1,post180,391,222276,3342290,67409,19056,5584


## 3. Что сохраняется

    - `data/slices/pre180.csv` и `data/slices/post180.csv` — списки произведений;
    - `results/corpus_summary.csv` — таблица объёма;
    - `results/pre180_frequencies.csv` и `results/post180_frequencies.csv` — частоты лемм.

    **Следующий шаг:** `01_train_models.ipynb`.

    Если вы измените корпус **после обучения**, удалите или перенесите старые `.model` перед новым запуском. Автоматической проверки изменений и контрольных сумм намеренно нет.